# Milestone 2: stationarity and identification

This notebook is generated by `tools/build_notebooks.py`. Edit the builder, not this file: a test fails if the two drift apart.

It follows course modules 5 and 6: test for a unit root, choose d, look at the ACF and PACF, and choose (p, q). Every number comes from the training split only, the first 90% of the course window, as ADR-0002 requires.

It reads three committed files in `app/public/data/`: `stationarity.json`, `acf_pacf.json` and `aic_grid.json`: test statistics, autocorrelations and AIC values. One cell near the end rebuilds two of them from the training rows of `prices.json` (committed too) and refits the chosen cell; it skips if that file is absent.

The pictures are in the app (`cd app && npm run dev`): the **Stationarity** page and the **AIC landscape** page.

In [ ]:
import collections

from arima_forecasting import selection, stationarity
from arima_forecasting.artifacts import find_app_data, read_artifact

app_data = find_app_data()
st = read_artifact(app_data, "stationarity")["data"]
ap = read_artifact(app_data, "acf_pacf")["data"]
grid = read_artifact(app_data, "aic_grid")["data"]
print("training rows:", st["n_train"], "| alpha:", st["alpha"])

## Module 5: is the log price stationary?

The course runs one test, ADF, on the whole series (train and test together), and draws a 12-day rolling mean and standard deviation. Here two tests with opposite null hypotheses run on the training rows only:

- **ADF**: the null is "a unit root". A small p rejects it, which is evidence for stationarity.
- **KPSS**: the null is "stationary". A small p rejects it, which is evidence against.

Each runs with `regression='c'` (the rule uses these) and again with `regression='ct'` (a linear trend; reported, never used by the rule). ADF picks its lag by AIC from 0 to `maxlag`. KPSS picks its bandwidth automatically. statsmodels clips the KPSS p-value to its table, so "<= 0.01" means "0.01 or smaller".

In [ ]:
def kpss_p(kpss):
    return {"smaller": "<= 0.01", "greater": ">= 0.10"}.get(kpss["p_value_bound"]) or f"{kpss['p_value']:.3f}"

for t in st["transforms"]:
    adf, kp = t["adf"], t["kpss"]
    print(f"{t['id']:>5} (d = {t['d']}, n = {t['n']}): ADF tau {adf['statistic']:.3f}, p {adf['p_value']:.3g}, lag {adf['used_lag']} of {adf['maxlag']}"
          f" | KPSS eta {kp['statistic']:.4f}, p {kpss_p(kp)}, {kp['nlags']} lags | {t['verdict']}")
for t in st["transforms"]:
    w = t["with_trend"]
    print(f"{t['id']:>5} with a trend: ADF p {w['adf']['p_value']:.3g}, KPSS eta {w['kpss']['statistic']:.4f} (p {kpss_p(w['kpss'])}) | {w['verdict']}")

## The decision

The 2 x 2 table: ADF rejects and KPSS does not, "stationary"; the reverse, "unit root"; both reject, "conflict"; neither, "inconclusive". The rule takes the smallest d whose verdict is "stationary" (ADR-0002, section 2). pmdarima's `ndiffs` gives its own answer with each of its tests, as a cross-check.

In [ ]:
decision = st["decision"]
print("d =", decision["d_selected"], "| tests_conflict:", decision["tests_conflict"])
print("re-applying the rule to the committed tests gives:", stationarity.decide_d(st["transforms"]))
nd = decision["pmdarima_ndiffs"]
print("pmdarima ndiffs:", {k: nd[k] for k in ("kpss", "adf", "pp")}, "agrees:", nd["agrees"])

## Module 6: ACF and PACF, with their bands

The course calls `plot_acf(x, lags=50)` and `plot_pacf(x, lags=50)`. The files use those plots' own settings: `fft=False` for the ACF and `method='ywm'` for the PACF. The shaded band is drawn around zero:

- ACF, Bartlett's formula: z * sqrt((1 + 2 * (r_1^2 + ... + r_(k-1)^2)) / n). It widens with the lag.
- PACF: z / sqrt(n) at every lag.

z = 1.96 for 95%. The next cell rebuilds the ACF band from the file's own ACF values and compares.

In [ ]:
import math

z = ap["z"]
for s in ap["series"]:
    r = [row["acf"] for row in s["rows"]]
    bartlett = [0.0] + [z * math.sqrt((1 + 2 * sum(x * x for x in r[1:k])) / s["n"]) for k in range(1, len(r))]
    gap = max(abs(a - row["acf_band"]) for a, row in zip(bartlett, s["rows"]))
    print(f"{s['transform']:>5}: lag-1 ACF {r[1]:.4f}, PACF band {s['rows'][1]['pacf_band']:.4f}, "
          f"ACF beyond its band at {s['acf_outside_count']} of {ap['nlags']} lags, PACF at {s['pacf_outside_count']}, "
          f"Bartlett rebuilt to within {gap:.1e}")

## Choosing p, q and the trend: AIC within one d

Every (p, q) from 0 to 5 is fitted at every d, with the trend terms of ADR-0002 section 4: a constant at d = 0, no drift and a drift at d = 1, none at d = 2. Nothing is dropped. A fit that raised is `failed`; one whose optimiser stopped early is `not_converged`; one with an inverse AR or MA root above 0.99 is `near_unit_root`. Only `ok` cells at the chosen d can be selected. AIC is never compared across d: d came from the tests above.

An ARMA likelihood can have several maxima, so each cell is fitted from several starts: statsmodels' default, each smaller model it nests padded with zeros, and seeded random starts. The highest likelihood is kept. A bigger model can then never score below a model it nests; the next cell checks that on the committed file.

In [ ]:
for row in grid["selection"]["status_counts"]:
    print(f"d = {row['d']}: " + ", ".join(f"{k} {row[k]}" for k in selection.STATUSES))
s = grid["selection"]
print("selected: ARIMA" + str(tuple(s["order"])), "trend", repr(s["trend"]), f"AIC {s['aic']:.3f}, {s['k_params']} parameters")
print("within 2 AIC of it:", [(w["p"], w["q"], w["trend"], round(w["delta_aic"], 3)) for w in s["within_2_aic"]])
b = s["bic_choice"]
print(f"BIC would choose: ({b['p']}, {s['d_selected']}, {b['q']}) trend {b['trend']!r}")
ranked = sorted((c for c in grid["cells"] if c["d"] == s["d_selected"] and c["aic"] is not None), key=lambda c: c["aic"])
for c in ranked[:6]:
    print(f"  ({c['p']}, {c['d']}, {c['q']}) {c['trend']}: AIC {c['aic']:.3f}, delta {c['delta_aic']:.3f}, {c['status']}, start: {c['start']}")
won = collections.Counter(c["start"].split(" (")[0].split(" ")[0].rstrip(",") for c in grid["cells"] if c["start"])
print("fits:", sum(c["starts_tried"] for c in grid["cells"]), "| the start that won:", dict(sorted(won.items())))
print("cells below a model they nest:", len(selection.nesting_violations(grid["cells"], grid["fit_settings"]["nesting_tolerance"])))

## pmdarima's stepwise search, as a cross-check

`auto_arima` runs twice on the same training rows: with the course's arguments, and with pmdarima's defaults. It is a local search with `maxiter=50`, and it hides its fits' warnings by default, so it can stop somewhere else. The gap compares its AIC with our cell of the same order and trend.

In [ ]:
check = grid["pmdarima_check"]
print("pmdarima:", check["status"], check["version"])
for run in check["runs"]:
    ag = run["agreement"]
    gap = "n/a" if ag["aic_gap_same_order"] is None else f"{ag['aic_gap_same_order']:+.5f}"
    print(f"{run['label']:>8}: ARIMA{tuple(run['order'])} intercept {run['with_intercept']}, AIC {run['aic']:.3f}, "
          f"converged {run['converged']} after {run['iterations']} iterations, {len(run['visited'])} fits kept | gap to our cell {gap}")

In [ ]:
# Rebuild the files from the training rows of prices.json, and refit the chosen cell.
import numpy as np
from arima_forecasting.artifacts import table_from_prices
from arima_forecasting.identification import acf_pacf_payload

if not (app_data / "prices.json").is_file():
    print("prices.json is not in this copy - skipped")
else:
    table = table_from_prices(read_artifact(app_data, "prices")["data"])
    train = table.loc[table["segment"] == "train", "log_close"].to_numpy()
    print("stationarity.json matches a fresh computation:", stationarity.stationarity_payload(train) == st)
    print("acf_pacf.json matches a fresh computation:", acf_pacf_payload(train) == ap)
    p, d, q = s["order"]
    # A cell won in the polish stage cannot be replayed from its label; its log-likelihood is
    # recomputed at the parameters the grid recorded. Any other cell is refitted from its start.
    key = (d, p, q, s["trend"])
    refit, _ = selection.refit_cell(train, grid["cells"], key)
    how = "refitted from its recorded start" if selection.replayable(grid["cells"], key) else "recomputed at its recorded parameters"
    print(f"the selected cell's log-likelihood, {how}, matches:", abs(refit["llf"] - next(c["llf"] for c in grid["cells"] if (c["d"], c["p"], c["q"], c["trend"]) == key)) < 1e-6)

## What comes next

Milestone 3 fits the selected model, and the course's refit beside it, and checks the residuals: are they white noise, and how far from normal are they?